# Warmup: Pandas to SQL, Same Brain, New Notation

**Module:** Week 4 Day 1 · **Format:** Individual, then partner check

This activity runs in two sittings today:

- **Part 1 (now, before any SQL):** answer six business questions in pandas, on your own. You have used every one of these moves since Week 2.
- **Part 2 (now):** export your DataFrame into a real SQLite database file.
- **Part 3 (after the SQL kickoff and DBeaver setup):** answer the same six questions again, in SQL, against the database you just created, and compare.

## Setup

Copy this notebook into your own workspace first (never work inside the provided lab folder):

```bash
mkdir -p student-work/week4/day1
cp "Week 4/Labs/Day 1/Warmup_Pandas_to_SQL.ipynb" student-work/week4/day1/
cd student-work/week4/day1
uv init
uv add pandas ipykernel
```

The `.venv` lands in `student-work/week4/day1/.venv`. In VS Code, select it as the notebook kernel (Select Kernel, Python Environments). If you see a `VIRTUAL_ENV does not match` warning, run `deactivate` first, and keep `.venv/` gitignored.

In [1]:
import pandas as pd

claims = pd.DataFrame(
    {
        "claim_id": ["C001", "C002", "C003", "C004", "C005", "C006", "C007", "C008"],
        "state": ["CT", "NY", "CT", "MA", "CT", "NY", "MA", "CT"],
        "claim_type": ["auto", "home", "auto", "home", "home", "auto", "auto", "auto"],
        "amount": [1200, 5400, 800, 3100, 9900, 450, 2200, None],
        "status": ["closed", "open", "open", "closed", "open", "closed", "open", "open"],
    }
)
claims

,claim_id,state,claim_type,amount,status
0,C001,CT,auto,1200.0,closed
1,C002,NY,home,5400.0,open
2,C003,CT,auto,800.0,open
3,C004,MA,home,3100.0,closed
4,C005,CT,home,9900.0,open
5,C006,NY,auto,450.0,closed
6,C007,MA,auto,2200.0,open
7,C008,CT,auto,NaN,open


## Part 1: Solve it in pandas

Write the pandas yourself for each question. No AI, no peeking at old notebooks unless stuck; this is retrieval practice.

### D1: The Connecticut book of business

An underwriter asks: "Show me only our Connecticut claims." 

In [7]:
# YOUR CODE HERE: filter claims to state == "CT"

claims[claims["state"]=="CT"]


,claim_id,state,claim_type,amount,status
0,C001,CT,auto,1200.0,closed
2,C003,CT,auto,800.0,open
4,C005,CT,home,9900.0,open
7,C008,CT,auto,NaN,open


### D2: Large claims for review

The claims manager wants the claim id and amount of every claim over 2,000 dollars, and nothing else.

In [13]:
# YOUR CODE HERE: rows where amount > 2000, columns claim_id and amount only

big_claims = claims[claims["amount"]>2000]
big_claims = big_claims[["claim_id", "amount"]]
big_claims

,claim_id,amount
1,C002,5400.0
3,C004,3100.0
4,C005,9900.0
6,C007,2200.0


### D3: Workload by state

Operations asks: "How many claims are we handling in each state?" 

In [ ]:
# YOUR CODE HERE: count claims per state
claims.groupby("state").size() #Switched from count to size


state
CT    4
MA    2
NY    2
dtype: int64

### D4: Average severity by product

Actuarial asks: "What is the average claim amount for auto versus home?" 

In [19]:
# YOUR CODE HERE: average amount per claim_type
claims.groupby("claim_type")["amount"].mean()

claim_type
auto    1162.500000
home    6133.333333
Name: amount, dtype: float64

### D5: The three biggest claims

Leadership wants the top three claims by amount, biggest first.

In [21]:
# YOUR CODE HERE: sort by amount descending, keep the top 3
claims.sort_values("amount", ascending= False).head(3)

,claim_id,state,claim_type,amount,status
4,C005,CT,home,9900.0,open
1,C002,NY,home,5400.0,open
3,C004,MA,home,3100.0,closed


### D6: The data quality catch

One claim has no recorded amount. Find it before finance does.

In [22]:
# YOUR CODE HERE: rows where amount is missing

claims[claims["amount"].isna()]


,claim_id,state,claim_type,amount,status
7,C008,CT,auto,NaN,open


## Part 2: Turn your DataFrame into a real database

Your first tiny pipeline of the week: pandas in, database out. The cell below creates `warmup_claims.db` in your current folder with one table named `claims`. This part is given.

In [23]:
import sqlite3

conn = sqlite3.connect("warmup_claims.db")
claims.to_sql("claims", conn, index=False, if_exists="replace")

# Prove the round trip works: read it back with SQL from Python.
pd.read_sql("SELECT * FROM claims", conn)

,claim_id,state,claim_type,amount,status
0,C001,CT,auto,1200.0,closed
1,C002,NY,home,5400.0,open
2,C003,CT,auto,800.0,open
3,C004,MA,home,3100.0,closed
4,C005,CT,home,9900.0,open
5,C006,NY,auto,450.0,closed
6,C007,MA,auto,2200.0,open
7,C008,CT,auto,NaN,open


Notice what happened to the missing amount: pandas showed `NaN`, the database stores `NULL`. Same idea, two spellings. That difference returns in D6's SQL version.

**Stop here.** The instructor kicks off SQL next: concepts, then a live demo. You will come back for Part 3 after DBeaver is set up.

## Part 3: Same six questions, now in SQL (after Activity 0)

1. In DBeaver, create a SQLite connection pointing at `student-work/week4/day1/warmup_claims.db`.
2. In `w4d1_sqlite_drills.sql`, under a `-- Warmup` header, write the SQL for D1 through D6.
3. Compare each result grid with your pandas output above. They should match row for row (SQL shows no index column, and rows have no order unless you ask for one).
4. With a partner: for each question, say the pandas out loud, then the SQL. Which clause had no pandas twin you could name?

## The point

You are not learning a new way of thinking this week. You are learning a second notation for a way of thinking you already own, plus the places where it runs: databases and warehouses that hold far more than a DataFrame comfortably can.